<a href="https://colab.research.google.com/github/mttomar/big-data-analytics/blob/main/pyspark_friend_recommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SQLContext

In [ ]:
sc = SparkContext.getOrCreate()
sqlContext = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Friends.txt to Friends (1).txt


In [ ]:
input_file = sc.textFile('Friends.txt')

input_file.take(10)

['0\t1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94',
 '1\t0,5,20,135,2409,8715,8932,10623,12347,12846,13840,13845,14005,20075,21556,22939,23520,28193,29724,29791,29826,30691,31232,31435,32317,32489,34394,35589,35605,35606,35613,35633,35648,35678,38737,43447,44846,44887,49226,49985,623,629,4999,6156,13912,14248,15190,17636,19217,20074,27536,29481,29726,29767,30257,33060,34250,34280,34392,34406,34418,34420,34439,34450,34651,45054,49592',
 '2\t0,117,135,1220,2755,12453,24539,24714,41456,45046,49927,6893,13795,16659,32828,41878',
 '3\t0,12,41,55,1532,12636,13185,27552,38737',
 '4\t0,8,14,15,18,27,72,80,15326,19068,19079,24596,42697,46126,74,77,33269,38792,38822',
 '5\t0,1,20,2022,22939,23527,30257,32503,35633,41457,43262,44846,49574,31140,32828',
 '6\t0,21,98,2203,32

In [ ]:
adj_list = input_file \
    .map(lambda line: line.strip().split("\t")) \
    .map(lambda x: (int(x[0]), [] if len(x) == 1 or x[1] == "" else [int(i) for i in x[1].split(",")]))

adj_list.take(5)

[(0,
  [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94]),
 (1,
  [0,
   5,
   20,
   135,
   2409,
   8715,
   8932,
   10623,
   12347,
   12846,
   13840,
   13845,
   14005,
   20075,
   21556,
   22939,
   23520,
   28193,
   29724,
   29791,
   29826,
   30691,
   31232,
   31435,
   32317,
   32489,
   34394,
   35589,
   35605,
   35606,
   35613,
   35633,
   35648,
   35678,
   38737,
   43

In [ ]:
from itertools import combinations

listA = adj_list.flatMap(
    lambda x: [((a, b), 1) for a, b in combinations(sorted(x[1]), 2)] +
              [((b, a), 1) for a, b in combinations(sorted(x[1]), 2)]
)

In [ ]:
listA.take(10)

[((1, 2), 1),
 ((1, 3), 1),
 ((1, 4), 1),
 ((1, 5), 1),
 ((1, 6), 1),
 ((1, 7), 1),
 ((1, 8), 1),
 ((1, 9), 1),
 ((1, 10), 1),
 ((1, 11), 1)]

In [ ]:
listA.count()

22909086

In [ ]:
listA_counts = listA.reduceByKey(lambda a, b: a + b)

In [ ]:
listA_counts.take(10)

[((26, 90), 1),
 ((43, 73), 1),
 ((43, 83), 1),
 ((45, 57), 1),
 ((68, 78), 1),
 ((82, 88), 1),
 ((87, 45), 1),
 ((20, 44846), 4),
 ((2409, 29767), 1),
 ((14248, 45054), 1)]

In [ ]:
listB = adj_list.flatMap(
    lambda x: [((x[0], f), 1) for f in x[1]]
)

In [ ]:
listB.take(10)

[((0, 1), 1),
 ((0, 2), 1),
 ((0, 3), 1),
 ((0, 4), 1),
 ((0, 5), 1),
 ((0, 6), 1),
 ((0, 7), 1),
 ((0, 8), 1),
 ((0, 9), 1),
 ((0, 10), 1)]

In [ ]:
filtered = listA_counts.subtractByKey(listB)

In [ ]:
filtered = filtered.cache()

In [ ]:
filtered.take(10)

[((75, 3), 1),
 ((85, 13), 1),
 ((56, 46), 1),
 ((21822, 27808), 1),
 ((6451, 4583), 1),
 ((15514, 7184), 1),
 ((16922, 24556), 1),
 ((33554, 36384), 1),
 ((34530, 34464), 6),
 ((34280, 34566), 4)]

In [ ]:
by_user = filtered.map(lambda x: (x[0][0], (x[0][1], x[1])))

In [ ]:
by_user.take(10)

[(75, (3, 1)),
 (85, (13, 1)),
 (56, (46, 1)),
 (21822, (27808, 1)),
 (6451, (4583, 1)),
 (15514, (7184, 1)),
 (16922, (24556, 1)),
 (33554, (36384, 1)),
 (34530, (34464, 6)),
 (34280, (34566, 4))]

In [ ]:
result = by_user.groupByKey().map(
    lambda x: (
        x[0],
        sorted(list(x[1]), key=lambda y: (-y[1], y[0]))[:10]
    )
)

In [ ]:
result.take(10)

[(56,
  [(57, 2),
   (59, 2),
   (60, 2),
   (61, 2),
   (1, 1),
   (2, 1),
   (3, 1),
   (4, 1),
   (5, 1),
   (6, 1)]),
 (34280,
  [(19431, 17),
   (34250, 14),
   (14047, 12),
   (14246, 12),
   (34326, 12),
   (11383, 11),
   (34263, 11),
   (34329, 11),
   (34439, 11),
   (7506, 10)]),
 (34200,
  [(19427, 32),
   (23616, 32),
   (34176, 32),
   (34364, 32),
   (11387, 31),
   (34205, 31),
   (13182, 30),
   (34326, 30),
   (14047, 29),
   (19428, 29)]),
 (35192,
  [(11767, 3),
   (24484, 3),
   (40910, 3),
   (41345, 3),
   (2206, 2),
   (4741, 2),
   (6466, 2),
   (6482, 2),
   (6508, 2),
   (6524, 2)]),
 (17636,
  [(23520, 22),
   (14027, 19),
   (13960, 18),
   (14040, 17),
   (13911, 16),
   (8932, 15),
   (14005, 15),
   (14044, 15),
   (13886, 14),
   (13965, 14)]),
 (43200,
  [(23550, 4),
   (42051, 4),
   (9194, 3),
   (15356, 3),
   (21556, 3),
   (25228, 3),
   (35633, 3),
   (1, 2),
   (129, 2),
   (2659, 2)]),
 (16880,
  [(16862, 7),
   (25228, 4),
   (28873, 4),
   (4

In [ ]:
formatted = result.map(
    lambda x: str(x[0]) + "\t" + ",".join([str(i[0]) for i in x[1]])
)

In [ ]:
formatted.take(10)

['56\t57,59,60,61,1,2,3,4,5,6',
 '34280\t19431,34250,14047,14246,34326,11383,34263,34329,34439,7506',
 '34200\t19427,23616,34176,34364,11387,34205,13182,34326,14047,19428',
 '35192\t11767,24484,40910,41345,2206,4741,6466,6482,6508,6524',
 '17636\t23520,14027,13960,14040,13911,8932,14005,14044,13886,13965',
 '43200\t23550,42051,9194,15356,21556,25228,35633,1,129,2659',
 '16880\t16862,25228,28873,49546,7521,13870,16867,16872,16905,16914',
 '25224\t27679,18916,27549,34299,10099,19,15186,21873,25200,27536',
 '8668\t8685,25186,25224,2588,7785,8671,8686,9471,10099,25256',
 '18604\t9777,12107,12637,12670,13929,15283,18297,18581,18609,35738']

In [ ]:
all_users = adj_list.map(lambda x: x[0])
users_with_recs = formatted.map(lambda line: int(line.split("\t")[0]))
missing_users = all_users.subtract(users_with_recs).map(lambda u: str(u) + "\t")
final_output = formatted.union(missing_users).sortBy(lambda x: int(x.split("\t")[0]))

In [ ]:
final_output.take(20)

['0\t38737,18591,27383,34211,337,352,1532,12143,12561,17880',
 '1\t35621,44891,14150,15356,35630,13801,13889,14078,25228,13805',
 '2\t41087,1,5,95,112,1085,1404,2411,3233,4875',
 '3\t27679,1,10,16,29,30,38,82,83,85',
 '4\t13,16,19,30,38,46,83,85,89,125',
 '5\t28193,29724,31232,34394,44887,2,1667,8715,12846,13840',
 '6\t9841,18581,6477,9463,9784,12577,12637,12667,18287,18299',
 '7\t12343,16539,40423,1,2,3,4,5,6,8',
 '8\t12,24,28,16,34211,14,15,18,27,29',
 '9\t82,4364,1,2,3,4,5,6,7,8',
 '10\t83,18,38,89,3,11,29,41,53,55',
 '11\t27552,7785,27573,27574,27589,27590,27600,27617,27620,27667',
 '12\t28,8,18,24,25186,25195,27552,27585,27590,27597',
 '13\t2572,38737,1,4,15,17,20,26,95,128',
 '14\t17438,19058,8,15,17,18,27,50,72,74',
 '15\t8,13,14,18,72,74,77,1,2,3',
 '16\t28,17,24,8,29,85,3,4,41,46',
 '17\t142,16,125,24,50,138,146,164,20565,20590',
 '18\t12,10,38,83,19044,8,14,15,27,28',
 '19\t14064,13800,13818,13967,1689,13795,13810,13898,14027,14043']

In [ ]:
output_lines = final_output.collect()

with open("recommendations.txt", "w") as f:
    for line in output_lines:
        f.write(line + "\n")

In [ ]:
required_users = {18, 87, 480, 512, 3971, 38283, 49772, 1152}

required_output = final_output.filter(
    lambda line: int(line.split("\t")[0]) in required_users
).collect()

for line in sorted(required_output, key=lambda x: int(x.split("\t")[0])):
    print(line)

18	12,10,38,83,19044,8,14,15,27,28
87	37356,10028,37188,19365,36908,30220,37448,40748,41903,44178
480	
512	488,489,491,492,497,499,504,505,506,508
1152	87,1079,1080,1081,1082,1083,1084,1085,1086,1087
3971	4046,22158,12403,4074,3958,10987,21958,3903,4163,22147
38283	1366,7024,4447,6962,23718,32387,5242,30749,5243,5249
49772	


In [ ]:
my_extra_user = 1142

extra_output = final_output.filter(
    lambda line: int(line.split("\t")[0]) == my_extra_user
).collect()

for line in extra_output:
    print(line)

1142	1085,1129,1688,7755,27555,27586,27590,27663,27667,29576
